In [ ]:
import pandas as pd
import numpy as np

In [ ]:
hdfc_data= pd.read_csv("/content/HDFC.csv")
hdfc_data

In [ ]:
news_data= pd.read_csv("/content/NewsData.io_HDFC.csv")
news_data

In [ ]:
# From 2022-04-01 to 2024-12-30

news_data["pubDate"]

In [ ]:
news_data['pubDate'] = news_data['pubDate'].str.strip('"').str.split().str[0]
news_data["pubDate"]

In [ ]:
news_data["sentiment"].value_counts()

In [ ]:
news_data["link"]

In [ ]:
!pip install newspaper3k transformers


In [ ]:
# preprocessing_sentiment.py
import re
from transformers import pipeline, AutoTokenizer
from collections import Counter

def clean_text(text):
    """Removes extra whitespace from the text."""
    return re.sub(r'\s+', ' ', text).strip()

#saved model
model_id = "/content/drive/MyDrive/finbert_kdave_trained"
# Loading the sentiment analysis pipeline and the tokenizer.
finbert = pipeline("sentiment-analysis", model=model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

#Splits the text into chunks that, when processed will be within the 512-token limit.
#Tokenizes the text (without special tokens), splits the token list into chunks and converts each chunk back to text.
def split_text_into_chunks(text, max_tokens=512):

    tokens = tokenizer.tokenize(text)
    chunk_size = max_tokens - 2
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[i:i+chunk_size]
        chunk_text = tokenizer.convert_tokens_to_string(chunk_tokens)
        chunks.append(chunk_text)
    return chunks

#Returns sentiment label and average score, splits into chunks if text is too long, mojority label selected by vote
def get_sentiment(text):
    text = clean_text(text)
    tokens = tokenizer.tokenize(text)
    if len(tokens) <= 510:
        result = finbert(text, truncation=True, max_length=512)
        return result[0]['label'], result[0]['score']

    # split into chunks if text is too long
    chunks = split_text_into_chunks(text, max_tokens=512)

    sentiments = []
    labels = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        result = finbert(chunk, truncation=True, max_length=512)
        labels.append(result[0]['label'])
        sentiments.append(result[0]['score'])

    if not sentiments:
        return None, None

    majority_label = Counter(labels).most_common(1)[0][0]
    avg_score = sum(sentiments) / len(sentiments)
    return majority_label, avg_score

#Runs the whole chain, returns a dict containing the sentiment result and document metadata.
def sentiment_chain(document):
    text = document.get("text", "")
    label, score = get_sentiment(text)
    return {
        "sentiment_label": label,
        "sentiment_score": score,
        "source": document.get("source", ""),
        "type": document.get("type", ""),
        "quarter": document.get("quarter", None)
    }


In [ ]:
!pip install lxml_html_clean

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
#from preprocessing_sentiment import sentiment_chain

# this is a function to extract article text using requests,BeautifulSoup
def extract_article_text(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Extracting all paragraphs using soup
        paragraphs = soup.find_all('p')
        text = ' '.join(p.get_text() for p in paragraphs)
        return text.strip()

    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return None

df = pd.read_csv('/content/NewsData.io_HDFC.csv')
df['sentiment_label'] = None
df['sentiment_score'] = None

# Iterate over all rows
for i, row in tqdm(df.iterrows(), total=len(df)):
    url = row['link'].strip('"')  # cleaning

    try:
        text = extract_article_text(url)
        if not text or len(text) < 50:
            raise ValueError("No meaningful content extracted.")

        document = {
            "text": text,
            "source": url,
            "type": "article",
            "quarter": None
        }

        result = sentiment_chain(document)
        df.at[i, 'sentiment_label'] = result['sentiment_label']
        df.at[i, 'sentiment_score'] = result['sentiment_score']

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        df.at[i, 'sentiment_label'] = "ERROR"
        df.at[i, 'sentiment_score'] = None

df.to_csv('articles_with_sentiment.csv', index=False)


In [ ]:
dataframe = pd.read_csv("/content/articles_with_sentiment.csv")

In [ ]:
len(dataframe)

In [ ]:
dataframe.columns

In [ ]:
final_df = dataframe.drop(["article_id","link","keywords","creator","description","content","pubDateTZ","image_url","video_url","source_id","source_name","source_priority","source_url","source_icon","language","country","category","sentiment","sentiment_stats","ai_tag","ai_region","ai_org","sentiment_label"], axis="columns")

In [ ]:
final_df

In [ ]:
#final_df['pubDate'] = final_df['pubDate'].str.strip('"').str.split().str[0]
final_df

In [ ]:
final_df.to_csv('final_sentiment_df.csv', index=False)

In [ ]:
df5 = pd.read_csv("/content/final_sentiment_df.csv")
df5

In [ ]:
df5.dropna()

In [ ]:
final_df.to_csv('Final_Sentiment_DF.csv', index=False)